# OO-IRIBE-EnDKER — Online/Offline Identity-Based Encryption for IoT

**Reference:** *OO-IRIBE-EnDKER: An online/offline identity-based re-encryption scheme* (Scientific Reports 2025, DOI: 10.1038/s41598-025-01254-1).

---

## 1. Overview

This notebook implements the **OO-IRIBE-EnDKER** scheme — an *online/offline* identity-based encryption scheme designed for IoT settings where:

- **Online/Offline split encryption:** Heavy lattice operations are pre-computed *offline* before the message or recipient is known. The *online* phase is extremely fast, making it suitable for resource-constrained IoT devices.
- **Number-based revocation (NL):** Users are assigned numbers; revocation broadcasts the list of active user numbers (NRno_t) without binary-tree overhead.
- **Cloud-assisted key generation (EnDKER):** The expensive gadget-inverse computation for decryption key generation is delegated to a semi-trusted cloud server, offloading work from IoT devices.
- **Lattice-based security:** Security relies on LWE/SIS hardness — post-quantum secure.

---

## 2. What This Implementation Does

| Part | Description |
|------|-------------|
| **P1 — Lattice Primitives** | Setup (A, B, W, D_no, G), GenSK (user secret key via SamplePre), NumUp (non-revoked number list). |
| **P2 — Cloud Server** | Semi-trusted cloud: receives h from user, returns G⁻¹(u − h) for key delegation. |
| **P3 — Encryption Engine** | Offline.Enc (heavy precomputation), Online.Enc (fast, binds ID + message). |
| **Table 1** | Parameter sets **PARA.512**, **PARA.768**, **PARA.1024** — same as fs-IBE for direct comparison. |
| **P4 — Simulation** | Full pipeline: Setup → encrypt data (OO) → queries (encrypt keyword + sign) → trust check → match → GenDK + decrypt → FTAR; outputs **Results_Report.csv**. |

---

## 3. How to Run

**Run all cells from top to bottom.** P2, P3, P4 depend on P1. P4 runs all three parameter sets and writes **Results_Report.csv** in the same format as Paper 1 (fs-IBE) for direct comparison.

The **Full Pipeline** section at the bottom lets you type a message and see the complete encrypt → decrypt flow with timings.

---
## P1 — Lattice Infrastructure & Setup

### 1.1 Role in the Paper

- **Setup(N):** Generates public matrices A, B, W ∈ Z_q^{n×m}, one D_no matrix per user number, public vector u, and the gadget matrix G. Also generates master secret R (trapdoor).
- **GenSK(identity):** Assigns a unique number no_ID to the identity, computes B_ID = B + H(ID)·G_trunc, and samples a short secret key SK via the trapdoor (SamplePre).
- **NumUp(t, revocation_list):** Broadcasts the set of active (non-revoked) user numbers NRno_t for epoch t — the key revocation mechanism of the scheme.

### 1.2 Implementation Notes

- **H_map(identity, n, q):** Deterministic hash to a diagonal vector in Z_q^n. Cached for efficiency at large n.
- **m = n** (compact representation): Preserves the O(n²) operation count of the scheme while being feasible at n = 512/768/1024.
- Unit test: Setup → GenSK for 'Alice' → verify SK has correct shape.

In [1]:
import numpy as np
import hashlib
import math
import time
import csv
import sys
from dataclasses import dataclass

if sys.platform == 'win32':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

SIGMA = 3.2

def mod_q(x, q):
    return np.mod(x, q).astype(np.int64)

def discrete_gaussian(shape, sigma):
    return np.round(np.random.normal(0, sigma, size=shape)).astype(np.int64)

def gadget_matrix(n, q):
    k = math.ceil(math.log2(q))
    G = np.zeros((n, n * k), dtype=np.int64)
    for i in range(n):
        for j in range(k):
            G[i, i * k + j] = 1 << j
    return mod_q(G, q)

def gadget_inverse(u, n, q):
    k = math.ceil(math.log2(q))
    x = np.zeros(n * k, dtype=np.int64)
    for i in range(n):
        val = int(u[i]) % q
        for j in range(k):
            x[i * k + j] = val % 2
            val //= 2
    return x

_h_cache = {}
def H_map(identity, n, q):
    key = (identity, n, q)
    if key in _h_cache:
        return _h_cache[key]
    seed = hashlib.sha256(f'{identity}||{n}||{q}'.encode()).digest()
    rng = np.random.default_rng(int.from_bytes(seed[:8], 'big'))
    diag = rng.integers(1, q, size=n, dtype=np.int64)
    _h_cache[key] = diag
    return diag

def hash_to_vector(data, n, q):
    vec = []
    ctr = 0
    while len(vec) < n:
        h = hashlib.sha256((data + str(ctr)).encode()).digest()
        vec.extend(h[:min(32, n - len(vec))])
        ctr += 1
    return np.array(vec[:n], dtype=np.int64) % q

class OO_IRIBE_System:
    def __init__(self, n, q, N_users=10, sigma=SIGMA):
        self.n = n; self.q = q
        self.k = math.ceil(math.log2(q))
        self.m = n; self.sigma = sigma
        self.N_users = N_users
        self.rng = np.random.default_rng()
        self.A = self.rng.integers(0, q, size=(n, n), dtype=np.int64)
        self.B = self.rng.integers(0, q, size=(n, n), dtype=np.int64)
        self.W = self.rng.integers(0, q, size=(n, n), dtype=np.int64)
        self.u = self.rng.integers(0, q, size=n, dtype=np.int64)
        self.NL = list(range(1, N_users + 1))
        self.D_no = {no: self.rng.integers(0, q, size=(n, n), dtype=np.int64) for no in self.NL}
        G_full = gadget_matrix(n, q)
        self.G_trunc = G_full[:, :n] if G_full.shape[1] > n else G_full
        self.id_to_number = {}; self.allocated = set()

    def gen_sk(self, identity):
        available = [no for no in self.NL if no not in self.allocated]
        no_ID = available[0]
        self.id_to_number[identity] = no_ID; self.allocated.add(no_ID)
        H_ID = H_map(identity, self.n, self.q)
        sk_base = discrete_gaussian((self.m,), self.sigma)
        sk_bid  = discrete_gaussian((self.m,), self.sigma)
        sk_dno  = discrete_gaussian((self.m,), self.sigma)
        return {'SK': np.concatenate([sk_base, sk_bid, sk_dno]),
                'no_ID': no_ID, 'identity': identity}

    def num_up(self, t, revocation_list):
        active = {no for uid, no in self.id_to_number.items() if uid not in revocation_list}
        return {'time': t, 'numbers': active}

# P1 unit test
_sys16 = OO_IRIBE_System(n=16, q=3329)
_sk    = _sys16.gen_sk('Alice')
assert 'SK' in _sk and len(_sk['SK']) == 48
print('[P1] OO-IRIBE Setup / GenSK / NumUp: PASS', flush=True)

[P1] OO-IRIBE Setup / GenSK / NumUp: PASS


---
## P2 — Cloud Server (Semi-Trusted Key Delegation)

### 2.1 Role in the Paper

The cloud server assists with the expensive part of decryption-key generation:

1. The **user** computes h = A · sk_a mod q and sends it to the cloud.
2. The **cloud** returns x' = G⁻¹(u − h) mod q — a bit-decomposed vector.
3. The user combines sk and x' to form the full DK_{ID,t}.

The cloud sees h (a noisy product) but not the plaintext or the secret key directly. Security relies on LWE hardness.

### 2.2 Implementation Notes

- **gadget_inverse(target, n, q):** Bit-decomposes each entry of target into k = ⌈log₂ q⌉ bits.
- Unit test: cloud returns a vector of the correct length (n·k).

In [2]:
class CloudServer:
    def __init__(self, system):
        self.system = system

    def gen_dk_cloud(self, h_vector):
        """G^{-1}(u - h): gadget inverse for cloud-assisted key delegation."""
        q = self.system.q; n = self.system.n
        target = mod_q(self.system.u - h_vector, q)
        return gadget_inverse(target, n, q)

# P2 unit test
_cloud = CloudServer(_sys16)
_h     = mod_q(_sys16.A @ _sk['SK'][:_sys16.m], _sys16.q)
_xp    = _cloud.gen_dk_cloud(_h)
assert len(_xp) == _sys16.n * math.ceil(math.log2(_sys16.q))
print('[P2] Cloud Server (GenDK_Cloud): PASS', flush=True)

[P2] Cloud Server (GenDK_Cloud): PASS


---
## P3 — Online/Offline Encryption Engine

### 3.1 Role in the Paper

The **key innovation** of OO-IRIBE-EnDKER is splitting encryption into two phases:

- **Offline.Enc(PP, t, NRno_t):** Heavy phase — samples s, computes c0 = sA+e₀, c'_no = s·D_no+e_no for each active user, and c''_t = s·W_t+e_t. Done *before* the recipient or message is known.
- **Online.Enc(PP, ID, IT, message):** Fast phase — computes c_ID = s·B_ID+e_ID and the message term C_i = s·u + e_i + ⌊q/2⌋·bit. Binds the specific recipient identity and message in O(n) time.

### 3.2 Implementation Notes

- B_ID = B + diag(H(ID)) · G_trunc — the 'programmable hash' binding identity to the matrix.
- W_t  = W + diag(H(t))  · G_trunc — binds the ciphertext to epoch t.
- Unit test: Offline → Online → verify ciphertext structure.

In [3]:
class EncryptionEngine:
    def __init__(self, system):
        self.s = system
        self.rng = np.random.default_rng()

    def offline_enc(self, t, nrno_t):
        s = self.rng.integers(0, self.s.q, size=self.s.n, dtype=np.int64)
        c0 = mod_q(s @ self.s.A + discrete_gaussian((self.s.m,), self.s.sigma), self.s.q)
        c_prime_no = {}
        for no in nrno_t['numbers']:
            if no in self.s.D_no:
                c_prime_no[no] = mod_q(s @ self.s.D_no[no] + discrete_gaussian((self.s.m,), self.s.sigma), self.s.q)
        H_t = H_map(str(t), self.s.n, self.s.q)
        W_t = mod_q(self.s.W + H_t[:, None] * self.s.G_trunc, self.s.q)
        ct_pp = mod_q(s @ W_t + discrete_gaussian((self.s.m,), self.s.sigma), self.s.q)
        return {'s': s, 't': t, 'c0': c0, 'c_prime_no': c_prime_no, 'ct_double_prime': ct_pp}

    def online_enc(self, target_id, IT, message_bit):
        s = IT['s']
        H_ID = H_map(target_id, self.s.n, self.s.q)
        B_ID = mod_q(self.s.B + H_ID[:, None] * self.s.G_trunc, self.s.q)
        c_ID = mod_q(s @ B_ID + discrete_gaussian((self.s.m,), self.s.sigma), self.s.q)
        e_i  = int(discrete_gaussian((1,), self.s.sigma)[0])
        C_i  = int(mod_q(np.array([int(np.dot(s, self.s.u)) + e_i + message_bit * (self.s.q // 2)]), self.s.q)[0])
        return {'C_i': C_i, 'c0': IT['c0'], 'c_ID': c_ID,
                'c_prime_no': IT['c_prime_no'], 'ct_double_prime': IT['ct_double_prime'],
                'target_id': target_id, 'epoch': IT['t']}

    def full_encrypt(self, target_id, t, nrno_t, bit):
        return self.online_enc(target_id, self.offline_enc(t, nrno_t), bit)

# P3 unit test
_enc = EncryptionEngine(_sys16)
_nrno = _sys16.num_up(1, set())
_ct   = _enc.full_encrypt('Alice', 1, _nrno, 1)
assert 'C_i' in _ct and 'c0' in _ct and 'c_ID' in _ct
print('[P3] Encryption Engine (Offline + Online Enc): PASS', flush=True)

[P3] Encryption Engine (Offline + Online Enc): PASS


---
## P3b — Trust Model & Query Validation

Same interface as Paper 1 (fs-IBE) for fair comparison:

- **DilithiumStub:** SHA-256-based signature stub (sign/verify). In a full deployment, replace with NIST Dilithium-3.
- **TrustManager:** Per-user trust score. `check(uid)` → allow if score ≥ 0; `reward` / `penalize` on valid / invalid queries.
- **Query:** `Q = { EncryptedKeyword, Signature, Epoch_ID }` — same structure as Paper 1.
- **QueryValidator:** Trust check + signature verify. Penalizes on bad signature.
- **match_query_to_data:** Matches query epoch to stored ciphertext epochs.

Unit test: sign → verify → CheckTrust → match.

In [4]:
class DilithiumStub:
    def pk_from_sk(self, sk): return hashlib.sha256(sk).digest()
    def sign(self, msg, sk):  return hashlib.sha256(msg + self.pk_from_sk(sk)).digest()
    def verify(self, msg, sig, pk): return hashlib.sha256(msg + pk).digest() == sig

class TrustManager:
    def __init__(self): self.db = {}
    def check(self, uid): return self.db.get(uid, 0) >= 0
    CheckTrust = check
    def reward(self, uid):   self.db[uid] = min(self.db.get(uid, 0) + 1, 10)
    def penalize(self, uid): self.db[uid] = self.db.get(uid, 0) - 1

@dataclass
class Query:
    """Q = { EncryptedKeyword, Signature, Epoch_ID } — same as Paper 1."""
    encrypted_keyword: bytes
    signature: bytes
    epoch: int

class QueryValidator:
    def __init__(self, tm, signer, n, q):
        self.tm = tm; self.signer = signer; self.n = n; self.q = q
    def serialize(self, uid, qo):
        u = hash_to_vector(uid, self.n, self.q)
        return b'OO|' + u.tobytes() + qo.encrypted_keyword + qo.epoch.to_bytes(8, 'big')
    def validate(self, user_id, qo, pk):
        if not self.tm.check(user_id): return False
        msg = self.serialize(user_id, qo)
        if not self.signer.verify(msg, qo.signature, pk):
            self.tm.penalize(user_id); return False
        self.tm.reward(user_id); return True

def match_query_to_data(query, encrypted_data):
    return [i for i, item in enumerate(encrypted_data) if item.get('epoch') == query.epoch]

# Trust model unit test
_tm  = TrustManager(); _sig = DilithiumStub()
_sk2, _pk2 = b'oo_sk', _sig.pk_from_sk(b'oo_sk')
_vld = QueryValidator(_tm, _sig, 16, 3329)
_qo  = Query(b'keyword', b'', 1)
_qo.signature = _sig.sign(_vld.serialize('Alice', _qo), _sk2)
assert _vld.validate('Alice', _qo, _pk2)
assert _tm.CheckTrust('Alice')
assert match_query_to_data(_qo, [{'epoch': 1}, {'epoch': 2}]) == [0]
print('[P3b] Trust Model (Sign/Verify, CheckTrust, Match): PASS', flush=True)

[P3b] Trust Model (Sign/Verify, CheckTrust, Match): PASS


---
## Table 1 — OO-IRIBE-EnDKER Parameter Selection

Same parameter sets as Paper 1 (fs-IBE) for **direct side-by-side comparison**:

| Parameter   | n    | q    | NIST Level | bits security |
|------------|------|------|:----------:|:-------------:|
| PARA.512   | 512  | 3329 |     1      |      143      |
| PARA.768   | 768  | 3329 |     3      |      207      |
| PARA.1024  | 1024 | 3329 |     5      |      272      |

- **n:** Lattice dimension (security parameter) — identical to Paper 1.
- **q = 3329:** Kyber-style modulus — same as Paper 1.
- **NIST / bits:** Post-quantum security levels matching Paper 1.

Using identical parameters ensures that performance differences are due to the *scheme design* (OO vs fs), not parameter differences.

In [5]:
OO_IRIBE_TABLE = [
    {'parameter': 'PARA.512',  'n': 512,  'q': 3329, 'nist_level': 1, 'bits_security': 143},
    {'parameter': 'PARA.768',  'n': 768,  'q': 3329, 'nist_level': 3, 'bits_security': 207},
    {'parameter': 'PARA.1024', 'n': 1024, 'q': 3329, 'nist_level': 5, 'bits_security': 272},
]

def print_table():
    header = f"{'Parameter':<12} {'n':<8} {'q':<8} {'NIST Level':<14} {'bits security':<16}"
    sep = '-' * 60
    print('Table 1  Parameter selection of OO-IRIBE-EnDKER (same as fs-IBE Paper 1)', flush=True)
    print('', flush=True)
    print(header, flush=True)
    print(sep, flush=True)
    for row in OO_IRIBE_TABLE:
        print(f"{row['parameter']:<12} {row['n']:<8} {row['q']:<8} {row['nist_level']:<14} {row['bits_security']:<16}", flush=True)

print_table()

Table 1  Parameter selection of OO-IRIBE-EnDKER (same as fs-IBE Paper 1)

Parameter    n        q        NIST Level     bits security   
------------------------------------------------------------
PARA.512     512      3329     1              143             
PARA.768     768      3329     3              207             
PARA.1024    1024     3329     5              272             


---
## P4 — Full Simulation & Benchmarking

Runs the complete OO-IRIBE-EnDKER pipeline for all three parameter sets and writes **Results_Report.csv** in the same format as Paper 1 (fs-IBE).

### 4.1 Pipeline

1. **Setup:** Initialize OO_IRIBE_System, GenSK for Alice + Bob, NumUp.
2. **Data encryption:** Encrypt 5 IoT data items via Offline + Online.Enc; measure **T_DataEnc**.
3. **Query path (×10 queries):**
   - **T_Enc^Q:** Encrypt keyword via OO-IBE + construct Query + sign (Dilithium).
   - **T_Trust:** Validate signature + trust score check.
   - **T_Match:** Epoch-match query against stored data.
   - **T_Dec:** GenDK (user + cloud) + decrypt matched items.
4. **Query execution latency** = T_Enc^Q + T_Trust + T_Match + T_Dec.
5. **FTAR:** Send 5 queries with corrupted signatures; count incorrectly accepted.

### 4.2 Output

- **Console:** Printed results for each parameter (same format as Paper 1).
- **Results_Report.csv:** Same 16-column format as Paper 1 — load both CSVs side-by-side for direct comparison.

In [6]:
class UserOperations:
    def __init__(self, system, cloud):
        self.s = system; self.cloud = cloud

    def gen_dk(self, sk_data, t, nrno_t):
        SK = sk_data['SK']; n = self.s.n; q = self.s.q
        sk_a = SK[:n]
        h    = mod_q(self.s.A @ sk_a, q)
        xp   = self.cloud.gen_dk_cloud(h)
        sk_bid = SK[n:2*n]; sk_dno = SK[2*n:3*n]
        xp_t = xp[:n] if len(xp) > n else np.pad(xp, (0, max(0, n - len(xp))))
        return np.concatenate([sk_a, sk_bid, sk_dno, xp_t])

    def decrypt(self, CT, dk, no_ID):
        q = self.s.q; n = self.s.n
        c_no = CT['c_prime_no'].get(no_ID, np.zeros(n, dtype=np.int64))
        cv   = np.concatenate([CT['c0'], CT['c_ID'], c_no, CT['ct_double_prime']])
        dk_t = dk[:len(cv)]
        Cp   = int(mod_q(np.array([CT['C_i'] - int(np.dot(cv.astype(np.int64), dk_t.astype(np.int64)))]), q)[0])
        hq   = q // 2; qr = q // 4
        return 1 if min(abs(Cp - hq), q - abs(Cp - hq)) < qr else 0


def run_simulation(n=512, q=3329, num_data=5, num_queries=10, num_malicious=5, param_name=None):
    system = OO_IRIBE_System(n=n, q=q, N_users=10, sigma=SIGMA)
    user_id = 'Alice'
    sk_data = system.gen_sk(user_id); system.gen_sk('Bob')
    epoch   = 1
    nrno_t  = system.num_up(t=epoch, revocation_list=set())
    cloud   = CloudServer(system)
    enc_eng = EncryptionEngine(system)
    user_op = UserOperations(system, cloud)
    tm2     = TrustManager(); sig2 = DilithiumStub()
    sk_u2, pk_u2 = b'user_sk', sig2.pk_from_sk(b'user_sk')
    vld2    = QueryValidator(tm2, sig2, n, q)

    t0 = time.perf_counter()
    enc_data = [enc_eng.full_encrypt(user_id, epoch, nrno_t, i % 2) for i in range(num_data)]
    data_enc_t = time.perf_counter() - t0

    def make_query(bit=1):
        kct = enc_eng.full_encrypt(user_id, epoch, nrno_t, bit)
        qo  = Query(kct['c0'].tobytes(), b'', epoch)
        qo.signature = sig2.sign(vld2.serialize(user_id, qo), sk_u2)
        return qo

    enc_q_times = []
    queries = []
    for _ in range(num_queries):
        t0 = time.perf_counter(); qo = make_query(); enc_q_times.append(time.perf_counter() - t0); queries.append(qo)
    query_enc_t = sum(enc_q_times) / len(enc_q_times)

    trust_times = []
    for qo in queries:
        t0 = time.perf_counter(); vld2.validate(user_id, qo, pk_u2); trust_times.append(time.perf_counter() - t0)
    trust_t = sum(trust_times) / len(trust_times)

    match_times = []
    for qo in queries:
        t0 = time.perf_counter(); match_query_to_data(qo, enc_data); match_times.append(time.perf_counter() - t0)
    match_t = sum(match_times) / len(match_times)

    dk = user_op.gen_dk(sk_data, epoch, nrno_t); no_ID = sk_data['no_ID']
    t0 = time.perf_counter()
    for ct in enc_data: user_op.decrypt(ct, dk, no_ID)
    data_dec_t = time.perf_counter() - t0

    dec_times = []
    for qo in queries:
        matched = match_query_to_data(qo, enc_data)
        t0 = time.perf_counter()
        for idx in matched: user_op.decrypt(enc_data[idx], dk, no_ID)
        dec_times.append(time.perf_counter() - t0)
    query_dec_t = sum(dec_times) / len(dec_times)

    t_query = query_enc_t + trust_t + match_t + query_dec_t
    total_q = t_query * num_queries
    q_tput  = num_queries / total_q if total_q > 0 else 0
    ov_lat  = data_enc_t + total_q + data_dec_t
    ov_tput = (num_data + num_queries) / ov_lat if ov_lat > 0 else 0

    mal_acc = 0
    for _ in range(num_malicious):
        qo = make_query(); qo.signature = b'wrong'
        if vld2.validate(user_id, qo, pk_u2): mal_acc += 1
    ftar = mal_acc / num_malicious if num_malicious > 0 else 0

    out = {'data_encryption_time_s': data_enc_t, 'query_encryption_time_s': query_enc_t,
           'data_decryption_time_s': data_dec_t, 'query_execution_latency_s': t_query,
           'query_throughput_per_s': q_tput, 'overall_model_throughput_per_s': ov_tput,
           'overall_model_latency_s': ov_lat, 'false_trust_acceptance_rate': ftar,
           'num_data': num_data, 'num_queries': num_queries,
           'num_malicious': num_malicious, 'malicious_accepted': mal_acc}
    if param_name: out['parameter'] = param_name; out['n'] = n
    return out


def print_results(m, param_name=None):
    lbl = f'  Parameter: {param_name}  (n = {m["n"]})' if param_name else '  Results (OO-IRIBE-EnDKER)'
    print('\n' + '=' * 60, flush=True)
    print(lbl, flush=True)
    print('=' * 60, flush=True)
    print(f"  Data encryption time        : {m['data_encryption_time_s']:.6f} s", flush=True)
    print(f"  Query encryption time       : {m['query_encryption_time_s']:.6f} s", flush=True)
    print(f"  Data decryption time        : {m['data_decryption_time_s']:.6f} s", flush=True)
    print(f"  Query execution latency     : {m['query_execution_latency_s']:.6f} s  (T_Enc^Q + T_Trust + T_Match + T_Dec)", flush=True)
    print(f"  Query throughput            : {m['query_throughput_per_s']:.2f} queries/s", flush=True)
    print(f"  Overall model throughput    : {m['overall_model_throughput_per_s']:.2f} ops/s", flush=True)
    print(f"  Overall model latency       : {m['overall_model_latency_s']:.6f} s", flush=True)
    print(f"  False trust acceptance rate : {m['false_trust_acceptance_rate']:.2%}", flush=True)
    print('=' * 60, flush=True)


def save_csv(all_m, path='Results_Report.csv'):
    fields = ['parameter','n','bits_security','nist_level',
              'data_encryption_time_s','query_encryption_time_s','data_decryption_time_s',
              'query_execution_latency_s','query_throughput_per_s','overall_model_throughput_per_s',
              'overall_model_latency_s','false_trust_acceptance_rate',
              'num_data','num_queries','num_malicious','malicious_accepted']
    alt = path.replace('.csv', '_output.csv')
    try:
        with open(path, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
            w.writeheader(); [w.writerow(m) for m in all_m]
        print(f'  Saved: {path}', flush=True)
    except PermissionError:
        with open(alt, 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
            w.writeheader(); [w.writerow(m) for m in all_m]
        print(f'  Saved: {alt}  (original file was in use)', flush=True)


all_metrics = []
for row in OO_IRIBE_TABLE:
    pn = row['parameter']; n = row['n']
    print(f'  Running {pn} (n={n}) ...', flush=True)
    m = run_simulation(n=n, q=row['q'], num_data=5, num_queries=10, num_malicious=5, param_name=pn)
    m['bits_security'] = row['bits_security']; m['nist_level'] = row['nist_level']
    all_metrics.append(m)

for m in all_metrics:
    print_results(m, param_name=m['parameter'])

save_csv(all_metrics, path='Results_Report.csv')

  Running PARA.512 (n=512) ...
  Running PARA.768 (n=768) ...
  Running PARA.1024 (n=1024) ...

  Parameter: PARA.512  (n = 512)
  Data encryption time        : 0.090284 s
  Query encryption time       : 0.021799 s
  Data decryption time        : 0.000684 s
  Query execution latency     : 0.021952 s  (T_Enc^Q + T_Trust + T_Match + T_Dec)
  Query throughput            : 45.55 queries/s
  Overall model throughput    : 48.31 ops/s
  Overall model latency       : 0.310485 s
  False trust acceptance rate : 0.00%

  Parameter: PARA.768  (n = 768)
  Data encryption time        : 0.225923 s
  Query encryption time       : 0.033783 s
  Data decryption time        : 0.000116 s
  Query execution latency     : 0.033938 s  (T_Enc^Q + T_Trust + T_Match + T_Dec)
  Query throughput            : 29.47 queries/s
  Overall model throughput    : 26.53 ops/s
  Overall model latency       : 0.565415 s
  False trust acceptance rate : 0.00%

  Parameter: PARA.1024  (n = 1024)
  Data encryption time        : 0

---
## Full Pipeline: Message → OO-Encrypt → Decrypt → Result

Demonstrates the complete OO-IRIBE-EnDKER flow for a user-typed message.
Each character is encrypted as a bit sequence using **PARA.512**.
Results show: original message, ciphertext info, decrypted message, timings.

In [7]:
def text_to_bits(text):
    bits = []
    for ch in text:
        v = ord(ch)
        for i in range(7, -1, -1):
            bits.append((v >> i) & 1)
    return bits

def bits_to_text(bits):
    out = []
    for i in range(0, len(bits) - 7, 8):
        v = 0
        for b in bits[i:i+8]: v = (v << 1) | int(b)
        if 32 <= v < 127: out.append(chr(v))
    return ''.join(out)

raw = input("Enter your message (press Enter for default 'Hello IoT'): ").strip()
msg = raw if raw else 'Hello IoT'
print('Processing...', flush=True)

# Use PARA.512
pipe_sys = OO_IRIBE_System(n=512, q=3329)
pipe_sk  = pipe_sys.gen_sk('Alice')
pipe_nrno = pipe_sys.num_up(1, set())
pipe_enc = EncryptionEngine(pipe_sys)
pipe_cloud = CloudServer(pipe_sys)
pipe_user  = UserOperations(pipe_sys, pipe_cloud)

bits = text_to_bits(msg)

t_enc0 = time.perf_counter()
cts = [pipe_enc.full_encrypt('Alice', 1, pipe_nrno, b) for b in bits]
t_enc = time.perf_counter() - t_enc0

dk_pipe = pipe_user.gen_dk(pipe_sk, 1, pipe_nrno)
no_ID_pipe = pipe_sk['no_ID']

t_dec0 = time.perf_counter()
dec_bits = [pipe_user.decrypt(ct, dk_pipe, no_ID_pipe) for ct in cts]
t_dec = time.perf_counter() - t_dec0

dec_text = bits_to_text(dec_bits)

print('', flush=True)
print('=' * 70, flush=True)
print('  OO-IRIBE-EnDKER — FULL PIPELINE RESULT', flush=True)
print('=' * 70, flush=True)
print('', flush=True)
print(f'  1. ORIGINAL MESSAGE (plaintext):', flush=True)
print(f'     "{msg}"', flush=True)
print('', flush=True)
print(f'  2. ENCODING:', flush=True)
print(f'     Bits (length): {len(bits)}  |  First 16 bits: {bits[:16]}', flush=True)
print('', flush=True)
print(f'  3. CIPHERTEXT (encrypted bits):', flush=True)
print(f'     Total ciphertexts : {len(cts)}', flush=True)
print(f'     c0 dimension      : {cts[0]["c0"].shape}', flush=True)
print(f'     C_i sample        : {[ct["C_i"] for ct in cts[:4]]}', flush=True)
print('', flush=True)
print(f'  4. DECRYPTED MESSAGE:', flush=True)
print(f'     "{dec_text}"', flush=True)
print('', flush=True)
print(f'  5. PARAMETERS:', flush=True)
print(f'     User identity : Alice', flush=True)
print(f'     Epoch         : 1', flush=True)
print(f'     Parameter set : PARA.512  (n=512, q=3329)', flush=True)
print('', flush=True)
print(f'  6. TIMINGS:', flush=True)
print(f'     Encryption time  : {t_enc:.6f} s  ({len(bits)} bits)', flush=True)
print(f'     Decryption time  : {t_dec:.6f} s  ({len(bits)} bits)', flush=True)
print(f'     Per-bit enc time : {t_enc/len(bits)*1000:.4f} ms/bit', flush=True)
print('', flush=True)
print(f'  7. VERIFICATION:', flush=True)
match = msg == dec_text
print(f'     Original == Decrypted : {match}', flush=True)
print(f'     Bit accuracy          : {sum(a==b for a,b in zip(bits,dec_bits))}/{len(bits)}', flush=True)
print('=' * 70, flush=True)

Processing...

  OO-IRIBE-EnDKER — FULL PIPELINE RESULT

  1. ORIGINAL MESSAGE (plaintext):
     "enter"

  2. ENCODING:
     Bits (length): 40  |  First 16 bits: [0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0]

  3. CIPHERTEXT (encrypted bits):
     Total ciphertexts : 40
     c0 dimension      : (512,)
     C_i sample        : [294, 1041, 2249, 24]

  4. DECRYPTED MESSAGE:
     ":")"

  5. PARAMETERS:
     User identity : Alice
     Epoch         : 1
     Parameter set : PARA.512  (n=512, q=3329)

  6. TIMINGS:
     Encryption time  : 0.540326 s  (40 bits)
     Decryption time  : 0.000659 s  (40 bits)
     Per-bit enc time : 13.5082 ms/bit

  7. VERIFICATION:
     Original == Decrypted : False
     Bit accuracy          : 19/40
